In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# Load the data
counts_file_name = "Filtered_Counts.csv"
links_file_name = "Journey_Links.geojson"

counts = pd.read_csv(counts_file_name)
links = gpd.read_file(links_file_name)

In [ ]:
#For some reason they were different types so i changed both to ints
counts['JOURNEY_LINK_ID'] = counts['JOURNEY_LINK_ID'].astype(int)
links['JOURNEY_LINK_ID']  = links['JOURNEY_LINK_ID'].astype(int)

#Merge the geometry from the links into the counts so we have the route for each journey link
merge = counts.merge(links[['JOURNEY_LINK_ID', 'geometry', 'SHAPESTLength']], on='JOURNEY_LINK_ID', how='left')

merge['DATE_TIME'] = pd.to_datetime(merge['DATE_TIME'])
merge.drop(columns=['JOURNEY_START', 'JOURNEY_END'], inplace=True)

merge['VEHICLE_DENSITY'] = merge['TOTAL_MATCHES'] / (merge['SHAPESTLength'] / 1000)  # Convert length to kilometers for density calculation

# Got values from statistics found in the shared document although need to double check
# Using these multipliers we can make sure we only account for non electric vehicles as they are what produce pollution
year_multipliers = {
    2023: 0.98,
    2024: 0.63
}

merge['VEHICLE_DENSITY_EV'] = merge['VEHICLE_DENSITY'] * merge['DATE_TIME'].dt.year.map(year_multipliers)

merge